# ROOTS — Step 1B: Independent Validation, Mutation, Protein Expression, and Target Shortlist (Python)

**Pipeline:** ROOTS — Receptor-oriented Oncotarget Screening of Natural Compounds Under Thermodynamic Simulation in Ovarian Cancer

**Runtime:** default Colab Python runtime (not R).

**Input:** `de_results.csv` from Step 1A, loaded automatically from Google Drive.

**What this notebook does:**
1. Loads the Step 1A differential expression table and applies primary candidate filtering.
2. Runs independent, cross-platform validation against GSE26712 (microarray).
3. Queries cBioPortal for TCGA-OV somatic mutation frequency on the validated candidates.
4. Applies a protein-level (Human Protein Atlas, immunohistochemistry) evidence gate.
5. Runs a fallopian-tube tissue-of-origin sanity check, given the ongoing debate over high-grade serous ovarian cancer's cell of origin.
6. Builds a composite-evidence ranked shortlist and saves it for Step 2.

Every intermediate is checkpointed to Drive as it is produced, so a disconnected runtime resumes from the last completed stage rather than recomputing from the start.

## 0. Environment setup, Drive mount, and checkpoint registry

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import json
from datetime import datetime

DRIVE_BASE = Path('/content/drive/MyDrive/ROOTS_Pipeline')
STEP1_DIR = DRIVE_BASE / 'Step1'
STEP2_DIR = DRIVE_BASE / 'Step2'
CHECKPOINT_DIR = STEP2_DIR / 'checkpoints'
for d in (STEP1_DIR, STEP2_DIR, CHECKPOINT_DIR, STEP1_DIR / 'logs'):
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive mounted: {Path('/content/drive/MyDrive').exists()}")
print(f"Step 1 input directory: {STEP1_DIR}")
print(f"Step 2 output directory: {STEP2_DIR}")
print(f"Timestamp: {datetime.now().isoformat()}")

# ---- Checkpoint registry: every intermediate this pipeline produces, and where it lives ----
CHECKPOINT_REGISTRY = {
    'de_results':    STEP1_DIR / 'de_results.csv',
    'primary_hits':  STEP1_DIR / 'primary_hits.csv',
    'primary_up':    STEP1_DIR / 'primary_up.csv',
    'pheno':         CHECKPOINT_DIR / 'pheno.csv',
    'geo_expr':      CHECKPOINT_DIR / 'geo_expr.csv',
    'geo_results':   CHECKPOINT_DIR / 'geo_results.csv',
    'geo_validated': CHECKPOINT_DIR / 'geo_validated.csv',
    'mutation_df':   CHECKPOINT_DIR / 'mutation_df.csv',
    'hpa_df':        CHECKPOINT_DIR / 'hpa_ihc_results.csv',
    'shortlist':     CHECKPOINT_DIR / 'shortlist.csv',
    'final_shortlist': CHECKPOINT_DIR / 'final_shortlist.csv',
}

def checkpoint_save(name, df):
    """Call immediately after building any dataframe worth keeping."""
    if name not in CHECKPOINT_REGISTRY:
        raise ValueError(f"'{name}' not in CHECKPOINT_REGISTRY — add it there first.")
    path = CHECKPOINT_REGISTRY[name]
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    ok = path.exists() and path.stat().st_size > 0
    print(f"{'✓' if ok else '✗ FAILED'} checkpoint_save('{name}') -> {path} "
          f"({len(df)} rows, {path.stat().st_size/1024:.1f} KB)")
    return ok

def checkpoint_load(name):
    """Loads a checkpoint into globals() if not already in memory. Returns True if available."""
    if name in globals() and globals()[name] is not None:
        return True
    path = CHECKPOINT_REGISTRY.get(name)
    if path and path.exists():
        globals()[name] = pd.read_csv(path)
        print(f"✓ Recovered '{name}' from checkpoint ({len(globals()[name])} rows)")
        return True
    print(f"✗ '{name}' not in memory and no checkpoint exists at {path}")
    return False

def checkpoint_status():
    """Single source of truth for pipeline state — safe to run any time."""
    print(f"{'='*70}\nCHECKPOINT STATUS — {datetime.now().isoformat()}\n{'='*70}")
    for name, path in CHECKPOINT_REGISTRY.items():
        in_memory = name in globals() and globals()[name] is not None
        on_disk = path.exists()
        if in_memory and on_disk:
            state = "✓ live + saved"
        elif in_memory and not on_disk:
            state = "⚠ live but UNSAVED — call checkpoint_save() now"
        elif on_disk and not in_memory:
            state = "○ saved, not loaded — checkpoint_load() will recover it"
        else:
            state = "✗ MISSING — must be rebuilt"
        print(f"  {name:20s} {state}")
    print(f"{'='*70}")

checkpoint_status()

## 1. Load Step 1A output and apply primary candidate filtering

Ensembl gene IDs are mapped to gene symbols in bulk via `mygene.info` — the symbol is what every downstream evidence source (GEO probe annotation, cBioPortal, HPA) actually keys on.

In [ ]:
de_results = pd.read_csv(STEP1_DIR / 'de_results.csv')
print(f"Loaded de_results: {de_results.shape[0]} rows, columns: {list(de_results.columns)}")

try:
    import mygene
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'mygene'], check=True)
    import mygene

mg = mygene.MyGeneInfo()

# Strip the Ensembl version suffix (ENSG00000223972.5 -> ENSG00000223972);
# mygene.info expects unversioned IDs
de_results['gene_id_clean'] = de_results['gene_id'].str.split('.').str[0]
unique_ids = de_results['gene_id_clean'].unique().tolist()
print(f"\nQuerying mygene.info for {len(unique_ids)} unique Ensembl IDs (batched automatically)...")

query_result = mg.querymany(unique_ids, scopes='ensembl.gene', fields='symbol', species='human', returnall=True)

id_to_symbol = {hit['query']: hit['symbol'] for hit in query_result['out'] if 'symbol' in hit}
print(f"Mapped {len(id_to_symbol)} / {len(unique_ids)} IDs to gene symbols")
if query_result['missing']:
    print(f"  {len(query_result['missing'])} IDs had no match (likely non-coding/deprecated)")

de_results['gene_symbol'] = de_results['gene_id_clean'].map(id_to_symbol)

# ---- Primary significance filter: |log2FC| >= 1, padj < 0.05 ----
LFC_THRESHOLD = 1.0
PADJ_THRESHOLD = 0.05

sig_mask = (
    (de_results['log2FoldChange'].abs() >= LFC_THRESHOLD) &
    (de_results['padj'] < PADJ_THRESHOLD) &
    (de_results['padj'].notna())
)
primary_hits = de_results[sig_mask].copy().sort_values('padj')
primary_up = primary_hits[primary_hits['log2FoldChange'] > 0].copy()

print(f"\nGenes passing DE threshold: {len(primary_hits)}")
print(f"  Upregulated (log2FC > 0):   {(primary_hits['log2FoldChange'] > 0).sum()}")
print(f"  Downregulated (log2FC < 0): {(primary_hits['log2FoldChange'] < 0).sum()}")
print(f"\nUpregulated candidates carried forward (therapeutic target screening focuses on")
print(f"genes overexpressed in tumour, i.e. inhibitable/druggable rather than genes lost): {len(primary_up)}")

checkpoint_save('primary_hits', primary_hits)
checkpoint_save('primary_up', primary_up)

## 2. Independent validation: GSE26712

**GEO series:** GSE26712 — 185 primary ovarian epithelial cancer + 10 normal ovarian epithelial samples, Affymetrix HG-U133A platform.

**Why cross-platform:** the original locked criteria specified platform-matched RNA-seq, but no large bulk RNA-seq ovarian cancer GEO dataset with a normal-tissue comparator exists. GSE26712 is the largest, most-cited OC GEO validation dataset. Cross-platform concordance (RNA-seq discovery, microarray validation) is in fact a *stronger* independence claim than within-platform concordance would be, since it rules out platform-specific measurement artefacts as the source of agreement.

**Concordance criterion:** same direction of change *and* nominal p < 0.05 (uncorrected) in GSE26712. The GEO cohort is not required to independently clear FDR correction — a deliberate, disclosed design choice reflecting its role as directional corroboration under a real power asymmetry (10 normal samples), not as an independent discovery cohort in its own right.

In [ ]:
try:
    import GEOparse
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'GEOparse'], check=True)
    import GEOparse

GEO_ACCESSION = "GSE26712"
print(f"Retrieving {GEO_ACCESSION} (this can take a few minutes)...")
gse = GEOparse.get_GEO(geo=GEO_ACCESSION, destdir="/tmp/geo_cache")
platform_id = list(gse.gpls.keys())[0]
pheno = gse.phenotype_data
checkpoint_save('pheno', pheno.reset_index())

from collections import OrderedDict
expr_dict = OrderedDict()
for gsm_name, gsm in gse.gsms.items():
    table = gsm.table
    if table is not None and len(table) > 0:
        expr_dict[gsm_name] = table.set_index('ID_REF')['VALUE'].astype(float)
geo_expr = pd.DataFrame(expr_dict)

gpl = gse.gpls[platform_id]
probe_to_gene = gpl.table[['ID', 'Gene Symbol']].dropna()
probe_to_gene = probe_to_gene[probe_to_gene['Gene Symbol'] != '']
geo_expr['gene_symbol'] = geo_expr.index.map(probe_to_gene.set_index('ID')['Gene Symbol'])
geo_expr = geo_expr.dropna(subset=['gene_symbol'])

# Where multiple probes map to the same gene, keep the highest-mean-expression probe
geo_expr['mean_expr'] = geo_expr.drop(columns=['gene_symbol']).mean(axis=1)
geo_expr = geo_expr.sort_values('mean_expr', ascending=False).drop_duplicates('gene_symbol')
geo_expr = geo_expr.set_index('gene_symbol').drop(columns=['mean_expr'])
checkpoint_save('geo_expr', geo_expr.reset_index())

print(f"\nGEO expression matrix: {geo_expr.shape[0]} genes x {geo_expr.shape[1]} samples")

In [ ]:
# ---- Sample classification via the actual GEO tissue-characteristics field ----
# A free-text keyword classifier ('normal'/'cancer' substring matching) systematically
# misses this series' actual phrasing and returns zero tumour samples. The correct field
# is the structured tissue characteristic, matched against its exact recorded values.
tissue_col = 'characteristics_ch1.0.tissue'
normal_samples = pheno.loc[pheno[tissue_col] == 'Normal ovarian surface epithelium'].index.tolist()
tumour_samples = pheno.loc[pheno[tissue_col] == 'Late-stage high-grade ovarian serous adenocarcinoma'].index.tolist()

tumour_samples = [s for s in tumour_samples if s in geo_expr.columns]
normal_samples = [s for s in normal_samples if s in geo_expr.columns]
print(f"Usable samples — Tumour: {len(tumour_samples)}, Normal: {len(normal_samples)}")

from scipy import stats

def geo_concordance_check(geo_expr_df, tumour_ids, normal_ids, candidate_genes, alpha=0.05):
    """Directional concordance of candidate genes in the GEO validation cohort.
    Welch's t-test (unequal variance), appropriate for microarray log-expression data."""
    results = []
    for gene in candidate_genes:
        if gene not in geo_expr_df.index:
            results.append({'gene': gene, 'geo_log2fc': np.nan, 'geo_pvalue': np.nan,
                             'geo_direction': 'not_found', 'concordant': False})
            continue
        tv = geo_expr_df.loc[gene, tumour_ids].values.astype(float)
        nv = geo_expr_df.loc[gene, normal_ids].values.astype(float)
        tv, nv = tv[~np.isnan(tv)], nv[~np.isnan(nv)]
        if len(tv) < 3 or len(nv) < 3:
            results.append({'gene': gene, 'geo_log2fc': np.nan, 'geo_pvalue': np.nan,
                             'geo_direction': 'insufficient_data', 'concordant': False})
            continue
        t_stat, p_val = stats.ttest_ind(tv, nv, equal_var=False)
        log2fc = np.mean(tv) - np.mean(nv)
        direction = 'up' if log2fc > 0 else 'down'
        results.append({'gene': gene, 'geo_log2fc': log2fc, 'geo_pvalue': p_val,
                         'geo_direction': direction, 'concordant': (direction == 'up') and (p_val < alpha)})
    return pd.DataFrame(results)

candidate_symbols = primary_up['gene_symbol'].dropna().unique().tolist()
print(f"Candidate gene symbols for GEO validation: {len(candidate_symbols)}")

geo_results = geo_concordance_check(geo_expr, tumour_samples, normal_samples, candidate_symbols)
checkpoint_save('geo_results', geo_results)

concordant_genes = geo_results[geo_results['concordant'] == True]
print(f"\nGEO concordance results:")
print(f"  Genes tested: {len(geo_results)}")
print(f"  Found in GEO: {(geo_results['geo_direction'] != 'not_found').sum()}")
print(f"  Concordant (same direction + p<0.05, uncorrected): {len(concordant_genes)}")
print(f"\nDISCLOSED DESIGN CHOICE: GEO concordance uses nominal p<0.05 (uncorrected),")
print(f"not FDR-corrected significance — directional corroboration under acknowledged power asymmetry.")

merged = primary_up.merge(geo_results, left_on='gene_symbol', right_on='gene', how='left')
geo_validated = merged[merged['concordant'] == True].copy()
print(f"\nCandidates surviving GEO validation: {len(geo_validated)}")
checkpoint_save('geo_validated', geo_validated)

## 3. Mutation frequency (cBioPortal, TCGA-OV)

Queried only for GEO-validated candidates — the set that has already cleared two independent lines of transcriptomic evidence. Capped at 50 genes per session to respect the public API's rate limits; the cap is generous relative to the typical size of a GEO-validated candidate set.

In [ ]:
CBIO_BASE = "https://www.cbioportal.org/api"
CBIO_STUDY = "ov_tcga_pan_can_atlas_2018"
CBIO_PROFILE = f"{CBIO_STUDY}_mutations"

def get_mutation_frequency(gene_symbol):
    try:
        gene_resp = requests.get(f"{CBIO_BASE}/genes/{gene_symbol}", timeout=30)
        if gene_resp.status_code != 200:
            return {'gene': gene_symbol, 'mutation_count': 0, 'sample_count': 0, 'mutation_freq': np.nan}
        entrez_id = gene_resp.json().get('entrezGeneId')
        if entrez_id is None:
            return {'gene': gene_symbol, 'mutation_count': 0, 'sample_count': 0, 'mutation_freq': np.nan}

        mut_resp = requests.post(
            f"{CBIO_BASE}/molecular-profiles/{CBIO_PROFILE}/mutations/fetch",
            json={"entrezGeneIds": [entrez_id], "sampleListId": f"{CBIO_STUDY}_all"},
            headers={"Accept": "application/json"}, timeout=30
        )
        if mut_resp.status_code != 200:
            return {'gene': gene_symbol, 'mutation_count': 0, 'sample_count': 0, 'mutation_freq': np.nan}

        mutations = mut_resp.json()
        unique_patients = set(m.get('patientId', '') for m in mutations)
        sample_resp = requests.get(f"{CBIO_BASE}/studies/{CBIO_STUDY}", headers={"Accept": "application/json"}, timeout=30)
        total = sample_resp.json().get('allSampleCount', 1) if sample_resp.status_code == 200 else 1

        return {'gene': gene_symbol, 'mutation_count': len(unique_patients), 'sample_count': total,
                'mutation_freq': len(unique_patients) / total if total > 0 else np.nan}
    except Exception as e:
        print(f"  cBioPortal error for {gene_symbol}: {e}")
        return {'gene': gene_symbol, 'mutation_count': 0, 'sample_count': 0, 'mutation_freq': np.nan}

target_genes = geo_validated['gene_symbol'].dropna().unique().tolist()
print(f"Querying cBioPortal for {len(target_genes)} genes (capped at 50 per session)...")

mutation_results = []
for i, gene in enumerate(target_genes[:50]):
    mutation_results.append(get_mutation_frequency(gene))
    if (i + 1) % 10 == 0:
        print(f"  Queried {i+1}/{min(len(target_genes), 50)}")
    time.sleep(0.5)  # rate limiting

mutation_df = pd.DataFrame(mutation_results)
checkpoint_save('mutation_df', mutation_df)
print(f"\nMutation data retrieved for {len(mutation_df)} genes")
if len(mutation_df) > 0:
    print(mutation_df.sort_values('mutation_freq', ascending=False).head(10).to_string(index=False))

## 4. Protein-level evidence: Human Protein Atlas ovarian IHC

The bulk HPA download supplies the Ensembl mapping needed to query each gene's per-gene IHC XML record directly. Within that record, the overall ovarian-tissue expression level is the direct `<level>` child of the ovary `<data>` block under the `technology="IHC"` `<tissueExpression>` element — not a per-cell-type `<level>` nested one layer deeper, which looks superficially similar but answers a different question.

In [ ]:
import zipfile, io, xml.etree.ElementTree as ET

print("Downloading HPA bulk file for symbol -> Ensembl mapping...")
hpa_resp = requests.get("https://www.proteinatlas.org/download/proteinatlas.tsv.zip", timeout=120)
z = zipfile.ZipFile(io.BytesIO(hpa_resp.content))
hpa_bulk_df = pd.read_csv(z.open(z.namelist()[0]), sep='\t')
symbol_to_ensembl = dict(zip(hpa_bulk_df['Gene'].str.upper(), hpa_bulk_df['Ensembl']))

target_genes = geo_validated['gene_symbol'].dropna().unique().tolist()
gene_ensembl_pairs = [(g, symbol_to_ensembl[str(g).upper()]) for g in target_genes if str(g).upper() in symbol_to_ensembl]
print(f"Resolved {len(gene_ensembl_pairs)} / {len(target_genes)} genes to Ensembl IDs")

def extract_ovary_ihc(root, gene_label, ensembl_id):
    """Overall (not per-cell-type) ovarian IHC expression level for one gene's HPA record."""
    tissue_blocks = root.findall('.//tissueExpression')
    ihc_blocks = [tb for tb in tissue_blocks if tb.get('technology') == 'IHC']
    if not ihc_blocks:
        return {'gene': gene_label, 'ensembl_id': ensembl_id, 'hpa_detected': 'NO_IHC_ASSAY', 'hpa_level': None}
    for block in ihc_blocks:
        for data_el in block.findall('data'):
            tissue_el = data_el.find('tissue')
            if tissue_el is not None and 'ovar' in (tissue_el.text or '').strip().lower():
                level_el = data_el.find('level')
                level_text = level_el.text.strip() if (level_el is not None and level_el.text) else None
                detected = level_text is not None and level_text.lower() != 'not detected'
                return {'gene': gene_label, 'ensembl_id': ensembl_id, 'hpa_detected': detected, 'hpa_level': level_text}
    return {'gene': gene_label, 'ensembl_id': ensembl_id, 'hpa_detected': 'IHC_PRESENT_NO_OVARY_ENTRY', 'hpa_level': None}

hpa_results = []
for i, (gene_label, ensembl_id) in enumerate(gene_ensembl_pairs):
    try:
        resp = requests.get(f"https://www.proteinatlas.org/{ensembl_id}.xml", timeout=30)
        if resp.status_code == 200:
            result = extract_ovary_ihc(ET.fromstring(resp.content), gene_label, ensembl_id)
        else:
            result = {'gene': gene_label, 'ensembl_id': ensembl_id, 'hpa_detected': f'FETCH_FAILED_{resp.status_code}', 'hpa_level': None}
    except Exception as e:
        result = {'gene': gene_label, 'ensembl_id': ensembl_id, 'hpa_detected': f'ERROR_{e}', 'hpa_level': None}
    hpa_results.append(result)
    if (i + 1) % 20 == 0:
        checkpoint_save('hpa_df', pd.DataFrame(hpa_results))
    time.sleep(0.5)

hpa_df = pd.DataFrame(hpa_results)
checkpoint_save('hpa_df', hpa_df)
print(f"\n{len(hpa_df)} genes processed")
print(hpa_df['hpa_detected'].value_counts())

## 5. Tissue-of-origin sanity check

High-grade serous ovarian cancer is increasingly understood to arise from the fallopian tube, not the ovarian surface epithelium itself. A candidate that is simply a normal fallopian-tube secretory or ciliated marker, rather than a tumour-specific signal, would be enriched here by construction — both cohorts compare tumour tissue against ovarian (not tubal) normal tissue. This flag does not exclude a candidate; it annotates it, since a fallopian-tube-origin marker overexpressed in HGSOC relative to ovarian surface epithelium can still be a biologically valid and translatable target.

In [ ]:
FALLOPIAN_TUBE_MARKERS = {
    "PAX8": "Canonical fallopian-tube secretory-cell marker; widely used HGSOC lineage marker",
    "WT1": "Expressed in fallopian tube epithelium and mesothelium; used in ovarian serous carcinoma diagnostics",
    "FOXJ1": "Ciliated epithelial cell marker; fallopian tube ciliated cells",
    "OVGP1": "Oviductal glycoprotein; fallopian tube secretory product",
    "CLDN4": "Claudin-4; documented expression in fallopian tube epithelium",
    "CD24": "Documented expression in fallopian tube epithelium and some HGSOC precursor lesions",
    "MSLN": "Mesothelin; expressed in mesothelial cells lining the fallopian tube serosa",
}

def tissue_origin_flag(gene_symbol):
    g = str(gene_symbol).upper()
    if g in FALLOPIAN_TUBE_MARKERS:
        return {'gene': gene_symbol, 'tissue_origin_flag': 'FLAGGED', 'tissue_origin_note': FALLOPIAN_TUBE_MARKERS[g]}
    return {'gene': gene_symbol, 'tissue_origin_flag': 'NOT_FLAGGED', 'tissue_origin_note': None}

all_target_genes = geo_validated['gene_symbol'].dropna().unique().tolist()
origin_df = pd.DataFrame([tissue_origin_flag(g) for g in all_target_genes])
n_flagged = (origin_df['tissue_origin_flag'] == 'FLAGGED').sum()
print(f"Tissue-of-origin check: {n_flagged} / {len(origin_df)} candidates flagged as known fallopian-tube markers")

## 6. Composite evidence score and final shortlist

In [ ]:
shortlist = geo_validated.copy()

shortlist = shortlist.merge(origin_df, left_on='gene_symbol', right_on='gene', how='left', suffixes=('', '_origin'))
shortlist.drop(columns=[c for c in ['gene_origin'] if c in shortlist.columns], inplace=True)
shortlist['is_fallopian_tube_marker'] = shortlist['tissue_origin_flag'] == 'FLAGGED'

shortlist = shortlist.merge(mutation_df[['gene', 'mutation_count', 'mutation_freq']],
                             left_on='gene_symbol', right_on='gene', how='left', suffixes=('', '_mut'))
shortlist.drop(columns=[c for c in ['gene_mut'] if c in shortlist.columns], inplace=True)

shortlist = shortlist.merge(hpa_df[['gene', 'hpa_detected', 'hpa_level']],
                             left_on='gene_symbol', right_on='gene', how='left', suffixes=('', '_hpa'))
shortlist.drop(columns=[c for c in ['gene_hpa'] if c in shortlist.columns], inplace=True)

# ---- Composite evidence score: one point per corroborating line of evidence,
# plus a small continuous contribution from mutation frequency ----
shortlist['evidence_score'] = 0.0
shortlist['evidence_score'] += shortlist['concordant'].apply(lambda x: 1 if x is True else 0)
shortlist['evidence_score'] += shortlist['hpa_detected'].apply(lambda x: 1 if x is True else 0)
shortlist['evidence_score'] += shortlist['mutation_freq'].fillna(0)

shortlist = shortlist.sort_values(['evidence_score', 'padj'], ascending=[False, True])
checkpoint_save('shortlist', shortlist)

TOP_N = 10
final_shortlist = shortlist.head(TOP_N).copy()

print(f"\n{'='*60}")
print(f"FINAL TARGET SHORTLIST (top {TOP_N} by composite evidence score)")
print(f"{'='*60}")
display_cols = [c for c in ['gene_symbol', 'log2FoldChange', 'padj', 'mutation_freq',
                             'hpa_detected', 'hpa_level', 'is_fallopian_tube_marker',
                             'tissue_origin_note', 'evidence_score'] if c in final_shortlist.columns]
print(final_shortlist[display_cols].to_string(index=False))

checkpoint_save('final_shortlist', final_shortlist)

## 7. Save outputs and provenance

In [ ]:
output_path = STEP1_DIR / "step1_target_shortlist.csv"
final_shortlist.to_csv(output_path, index=False)
print(f"Target shortlist saved to: {output_path}")

full_output = STEP1_DIR / "step1_all_candidates.csv"
shortlist.to_csv(full_output, index=False)
print(f"Full GEO-validated candidate table saved to: {full_output}")

(STEP1_DIR / "logs").mkdir(parents=True, exist_ok=True)
provenance = {
    'timestamp': datetime.now().isoformat(),
    'geo_validation_series': 'GSE26712',
    'geo_platform': 'Affymetrix HG-U133A (cross-platform validation)',
    'geo_concordance_criterion': 'same direction AND nominal p<0.05 (uncorrected)',
    'cbio_study': CBIO_STUDY,
    'hpa_source': 'https://www.proteinatlas.org/{ENSEMBL_ID}.xml, IHC technology, ovarian tissue block',
    'candidates_after_de': len(primary_up),
    'candidates_after_geo': len(geo_validated),
    'final_shortlist_size': len(final_shortlist),
    'fallopian_tube_markers_flagged': int(n_flagged),
    'scope_boundaries': [
        'CNV excluded from prioritisation (compute cost under zero-cost constraint)',
        'Expression-primary, mutation-supported prioritisation, not full multi-omic integration',
    ],
    'confound_disclosure': 'TCGA/GTEx batch perfectly confounded with condition in the Step 1A discovery cohort; GSE26712 provides a differently-confounded, independent second line of evidence',
}

with open(STEP1_DIR / "logs" / "step1b_provenance.json", 'w') as f:
    json.dump(provenance, f, indent=2)

print(f"\n=== Step 1B complete. Proceed to Step 2 ===")
print(f"Step 2 will load: {output_path}")

## Completion

The following files are now on Google Drive at `ROOTS_Pipeline/Step1/`:
- `step1_target_shortlist.csv` — top-10 ranked targets by composite evidence score (input to Step 2)
- `step1_all_candidates.csv` — the full GEO-validated candidate table, unranked cutoff
- `primary_hits.csv`, `primary_up.csv` — intermediate DE-filtered tables
- `logs/step1b_provenance.json` — thresholds, data sources, disclosed design choices

**Next:** Step 2 (structural characterisation) reads `step1_target_shortlist.csv` directly.